# Platypus: a pretrained reader in the pipeline

Does a reader on a small pretrained model's states (better ranking with true labels: AUC 0.77-0.80 vs 0.72) give the package more RIGHT written inferences and more right CONFIRMED ones than the word-count reader, on the same 20,000 labelled records? `tools/confirm_pre.py`, about 15-25 minutes a model on an A100 (any GPU works).

**Secrets:** nothing here needs a key; the models are public. Never paste a key into this notebook.

Copyright Christian Bird. Apache 2.0.

In [ ]:
REPO = 'https://github.com/chrisbirduk/platypus.git'
BRANCH = 'cg27a-distill'
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/platypus'
MODELS = ['gpt2', 'Qwen/Qwen2.5-0.5B']   # one run each; gpt2 first (the cheapest reader)
DOCS = 50000           # distant documents (as tools/confirm_min.py)


In [ ]:
import subprocess, torch, os
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive'); os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.isdir('/content/platypus/.git'):
    !git clone -q --branch {BRANCH} {REPO} /content/platypus
else:
    !git -C /content/platypus fetch -q origin {BRANCH} && git -C /content/platypus checkout -q {BRANCH} && git -C /content/platypus reset -q --hard origin/{BRANCH}
%cd /content/platypus
!git log -1 --oneline
!pip -q install transformers scikit-learn

## The distant set (DocRED's distant training documents, 91 MB, checked against its SHA-256)

In [ ]:
import hashlib, shutil, urllib.request
DIST_URL = 'https://media.githubusercontent.com/media/multiful/Information_Extraction/main/docred_data/data/train_distant.json.gz'
DIST_SHA = 'c420c0429310583cfc9459f7daa26b1f4c11ff5c7a1481aa64ab9db2b296b905'
os.makedirs('/content/distant', exist_ok=True); dst = '/content/distant/train_distant.json.gz'
cache = f'{DRIVE_DIR}/train_distant.json.gz' if USE_DRIVE else dst
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
if not (os.path.exists(cache) and sha(cache) == DIST_SHA):
    urllib.request.urlretrieve(DIST_URL, cache)
assert sha(cache) == DIST_SHA, 'distant set: checksum mismatch'
if cache != dst: shutil.copy(cache, dst)
print('distant set ready:', dst, os.path.getsize(dst) // 2**20, 'MB')


## Run

In [ ]:
import shutil
for MODEL in MODELS:
    os.environ.update(CP_DOCS=str(DOCS), CP_MODEL=MODEL, CP_LAYER='mid')
    log = f"/content/confirm_pre_{MODEL.replace('/', '_')}.log"
    !python tools/confirm_pre.py /content/distant/train_distant.json.gz 2>&1 | grep -v 'Loading weights' | tee {log}
    if USE_DRIVE: shutil.copy(log, DRIVE_DIR + '/' + os.path.basename(log))
